In [2]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import zipfile

def download_afip_data(years: list, base_output_dir: str):
    """
    Scrapes the AFIP aggregated data page and downloads zip files for the specified years.
    Skips files that are already fully and successfully downloaded.
    """
    base_url = "https://www.afip.gob.ar"
    page_url = f"{base_url}/operadoresComercioExterior/informacionAgregada/informacion-agregada.asp"
    
    years = [str(y) for y in years]
    print(f"Fetching main page: {page_url}")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    
    try:
        response = requests.get(page_url, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch the website: {e}")
        return
    
    soup = BeautifulSoup(response.text, 'html.parser')
    all_links = soup.find_all('a', href=True)
    download_targets = []
    
    for link in all_links:
        href = link['href']
        if "download.aspx?filename=" in href:
            if any(f"filename={year}" in href for year in years):
                download_targets.append(href)
                
    if not download_targets:
        print("No files found for the requested years.")
        return
        
    print(f"Found {len(download_targets)} files to download.")
    
    for href in download_targets:
        file_url = urljoin(base_url, href)
        filename = href.split("filename=")[-1]
        
        year_match = re.match(r"^(\d{4})", filename)
        
        if year_match:
            file_year = year_match.group(1)
            year_dir = os.path.join(base_output_dir, file_year)
            os.makedirs(year_dir, exist_ok=True)
            
            save_path = os.path.join(year_dir, filename)
            
            if os.path.exists(save_path):
                # Check if the file is actually a valid, uncorrupted zip file
                if zipfile.is_zipfile(save_path):
                    print(f"Skipping {filename} - valid zip already exists in {year_dir}")
                    continue
                else:
                    print(f"Found broken/incomplete file for {filename}. Redownloading...")
            # ---------------------------
                
            print(f"Downloading {filename} to {year_dir}/ ...")
            
            try:
                with requests.get(file_url, headers=headers, stream=True) as r:
                    r.raise_for_status()
                    # 'wb' will overwrite the broken file if one existed
                    with open(save_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            f.write(chunk)
            except requests.exceptions.RequestException as e:
                print(f"Failed to download {filename}: {e}")
        else:
            print(f"Could not extract year from filename: {filename}. Skipping.")
                    
    print("\nProcess complete!")

In [3]:
# --- Execution ---
download_afip_data(years=[2022], base_output_dir="../data/raw/")

Fetching main page: https://www.afip.gob.ar/operadoresComercioExterior/informacionAgregada/informacion-agregada.asp
Found 12 files to download.
Skipping 202212.zip - valid zip already exists in ../data/raw/2022
Skipping 202211.zip - valid zip already exists in ../data/raw/2022
Found broken/incomplete file for 202210.zip. Redownloading...

Process complete!


In [4]:
# --- Execution ---
download_afip_data(years=[2021], base_output_dir="../data/raw/")

Fetching main page: https://www.afip.gob.ar/operadoresComercioExterior/informacionAgregada/informacion-agregada.asp
Found 12 files to download.

Process complete!


In [5]:
# --- Execution ---
download_afip_data(years=[2020], base_output_dir="../data/raw/")

Fetching main page: https://www.afip.gob.ar/operadoresComercioExterior/informacionAgregada/informacion-agregada.asp
Found 12 files to download.

Process complete!


In [6]:
# --- Execution ---
download_afip_data(years=[2019], base_output_dir="../data/raw/")

Fetching main page: https://www.afip.gob.ar/operadoresComercioExterior/informacionAgregada/informacion-agregada.asp
Found 12 files to download.

Process complete!


In [7]:
# --- Execution ---
download_afip_data(years=[2018], base_output_dir="../data/raw/")

Fetching main page: https://www.afip.gob.ar/operadoresComercioExterior/informacionAgregada/informacion-agregada.asp
Found 12 files to download.

Process complete!


In [8]:
# --- Execution ---
download_afip_data(years=[2017], base_output_dir="../data/raw/")

Fetching main page: https://www.afip.gob.ar/operadoresComercioExterior/informacionAgregada/informacion-agregada.asp
Found 11 files to download.

Process complete!
